# Experiment 1: Reference Coupling Model

**Goal**: Quantify how each spectral descriptor depends on pitch (F0) and
pitch dynamics (velocity, acceleration) in the reference violin recordings.

For each of the 7 spectral descriptors we fit two OLS regressions:

- **Model A** (static): $d_i \sim \beta_0 + \beta_1 f_0 + \beta_2 f_0^2$
- **Model B** (with dynamics): $d_i \sim \beta_0 + \beta_1 f_0 + \beta_2 f_0^2 + \beta_3 \dot{f}_0 + \beta_4 \ddot{f}_0$

We record $R^2$, $\Delta R^2$, F-test, and all coefficient p-values.
Model B coefficients define the **reference coupling function** $g_i^{\text{ref}}$
used in Experiment 2.

**Inputs**: Pre-extracted feature parquet from the feature extraction notebook.

**Outputs**: `artifacts/evaluation/exp1_ref_coupling_models.npz`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation.pitch_metrics import PitchMetrics
from visualize import plot_coupling_scatter

print(f"Project root: {PROJECT_ROOT}")

/home/namkhanh/miniconda3/envs/conda_env3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-09 01:39:13.956394: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-09 01:39:14.257230: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 01:39:15.481184: W tensorflow/compiler/xla/stream_executor/platfo

Project root: /home/namkhanh/Project/final_project/src


/home/namkhanh/miniconda3/envs/conda_env3.10/lib/python3.10/site-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Configuration

In [2]:
# ── Input: pre-extracted features (from feature_extraction notebook) ──
REF_PARQUET = PROJECT_ROOT / "data" / "processed" / "bach_violin_timbre_features.parquet"

# ── Analysis parameters ───────────────────────────────────────────────
SR          = 16_000
FRAME_WIDTH = 0.25      # must match what was used during extraction
OVERLAP     = 0.0
CONFIDENCE_THRESHOLD = 0.85
MEDIAN_WINDOW = 5       # median filter width for F0 before derivatives

# Descriptors to analyse (must exist in the parquet)
DESCRIPTORS = [
    "spectral_centroid",
    "spectral_crest",
    "spectral_decrease",
    "spectral_flatness",
    "spectral_roll_off",
    "spectral_skewness",
    "spectral_spread",
]

# ── Output ─────────────────────────────────────────────────────────────
OUT_DIR = PROJECT_ROOT / "artifacts" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reference data: {REF_PARQUET}  (exists: {REF_PARQUET.exists()})")
print(f"Descriptors:    {DESCRIPTORS}")

Reference data: /home/namkhanh/Project/final_project/src/data/processed/bach_violin_timbre_features.parquet  (exists: False)
Descriptors:    ['spectral_centroid', 'spectral_crest', 'spectral_decrease', 'spectral_flatness', 'spectral_roll_off', 'spectral_skewness', 'spectral_spread']


## 1. Load reference features

In [3]:
df_ref = pd.read_parquet(REF_PARQUET)
print(f"Loaded {len(df_ref):,} frames, columns: {list(df_ref.columns)}")

# Verify required columns exist
missing = [d for d in DESCRIPTORS + ["f0_hz", "f0_confidence"] if d not in df_ref.columns]
if missing:
    raise ValueError(f"Missing columns in parquet: {missing}")

df_ref.head()

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Pandas requires version '10.0.1' or newer of 'pyarrow' (version '9.0.0' currently installed).
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

## 2. Prepare feature table

Build the per-frame matrix: F0, F0 derivatives, 7 descriptors.  
Keep only voiced, high-confidence, finite rows.

In [ ]:
f0_raw = df_ref["f0_hz"].to_numpy(dtype=np.float64)
conf   = df_ref["f0_confidence"].to_numpy(dtype=np.float64)

# Apply confidence threshold
f0 = f0_raw.copy()
f0[conf < CONFIDENCE_THRESHOLD] = np.nan

# Compute derivatives
dt = FRAME_WIDTH * (1.0 - OVERLAP)  # seconds between frames
f0_dot, f0_ddot = PitchMetrics.f0_dynamics(f0, dt=dt, median_window=MEDIAN_WINDOW)

# Build feature matrix
features_2d = df_ref[DESCRIPTORS].to_numpy(dtype=np.float64)

# Voiced mask: F0 > 0, finite, and all descriptors finite
voiced = (f0 > 0) & np.isfinite(f0) & np.all(np.isfinite(features_2d), axis=1)

print(f"Total frames:  {len(f0):,}")
print(f"Voiced frames: {voiced.sum():,}  ({100 * voiced.mean():.1f}%)")
print(f"dt = {dt:.4f} s")

## 3. Fit coupling models (Model A + Model B)

In [ ]:
ref_models = PitchMetrics.fit_coupling_model(
    features_2d, f0, f0_dot, f0_ddot, DESCRIPTORS,
)

print(f"Fitted models for {len(ref_models)} descriptors.")

## 4. Results table

In [ ]:
COEFF_NAMES_A = ["\u03b20", "\u03b21(f0)", "\u03b22(f0\u00b2)"]
COEFF_NAMES_B = COEFF_NAMES_A + ["\u03b23(f0_dot)", "\u03b24(f0_ddot)"]

rows = []
for desc in DESCRIPTORS:
    m = ref_models[desc]
    rows.append({
        "descriptor": desc,
        "R2_A": m["r2_a"],
        "R2_B": m["r2_b"],
        "delta_R2": m["delta_r2"],
        "F_stat": m["f_stat"],
        "F_pvalue": m["f_pvalue"],
        **{f"B_{name}": coeff for name, coeff in zip(COEFF_NAMES_B, m["coeffs_b"])},
        **{f"B_{name}_p": pv for name, pv in zip(COEFF_NAMES_B, m["coeff_pvalues_b"])},
    })

df_results = pd.DataFrame(rows).set_index("descriptor")

print("=== R\u00b2 and F-test ===")
display(df_results[["R2_A", "R2_B", "delta_R2", "F_stat", "F_pvalue"]].round(6))

print("\n=== Model B coefficients ===")
coeff_cols = [c for c in df_results.columns if c.startswith("B_") and not c.endswith("_p")]
display(df_results[coeff_cols].round(6))

print("\n=== Model B coefficient p-values ===")
pval_cols = [c for c in df_results.columns if c.endswith("_p")]
display(df_results[pval_cols].round(6))

## 5. Scatter plots with regression overlay

In [ ]:
plot_data = {
    "reference": {
        "f0": f0,
        "features_2d": features_2d,
        "feat_keys": DESCRIPTORS,
        "models": ref_models,
    }
}

plot_coupling_scatter(
    plot_data, DESCRIPTORS,
    save_path=str(FIG_DIR / "exp1_coupling_scatter.png"),
)

## 6. Descriptor–F0 correlation summary

In [ ]:
ref_corr = PitchMetrics.correlation_summary(features_2d, f0, DESCRIPTORS)

df_corr = pd.DataFrame(ref_corr).T
df_corr.index.name = "descriptor"
display(df_corr.round(4))

## 7. Save reference models

In [ ]:
import pickle

# Save the full model dict (coefficients + stats) for Experiment 2
model_path = OUT_DIR / "exp1_ref_coupling_models.pkl"
with open(model_path, "wb") as fh:
    pickle.dump({
        "models": ref_models,
        "descriptors": DESCRIPTORS,
        "config": {
            "sr": SR,
            "frame_width": FRAME_WIDTH,
            "overlap": OVERLAP,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "median_window": MEDIAN_WINDOW,
            "dt": dt,
        },
        "correlation": ref_corr,
    }, fh)
print(f"Saved reference models: {model_path}")

# Also save the results table
csv_path = OUT_DIR / "exp1_ref_coupling_results.csv"
df_results.to_csv(csv_path)
print(f"Saved results CSV:     {csv_path}")